# Fact Flights Build

Builds the flight-grain fact table from the standardized BTS On-Time dataset.

## Grain
One row per reported flight record.

## Conformed Dimensions
- Date
- BTS Airline
- Origin Airport
- Destination Airport
- Route

## Output
`fact_flights`

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, upper, trim
from pyspark.sql.window import Window

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 3, Finished, Available, Finished, False)

In [2]:
flights = spark.table("silver_flights")

dim_date = spark.table("dim_date")
dim_airline = spark.table("dim_airline")
dim_airport = spark.table("dim_airport")
dim_route = spark.table("dim_route")

print("Silver Flight Rows:", flights.count())
print("Date Dimension Rows:", dim_date.count())
print("Airline Dimension Rows:", dim_airline.count())
print("Airport Dimension Rows:", dim_airport.count())
print("Route Dimension Rows:", dim_route.count())

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 4, Finished, Available, Finished, False)

Silver Flight Rows: 547271
Date Dimension Rows: 31
Airline Dimension Rows: 155
Airport Dimension Rows: 6072
Route Dimension Rows: 11920


In [3]:
date_lookup = (
    dim_date
    .select(
        "date_key",
        "flight_date"
    )
)

airline_lookup = (
    dim_airline
    .select(
        "airline_key",
        "unique_carrier"
    )
)

origin_airport_lookup = (
    dim_airport
    .select(
        col("airport_key").alias("origin_airport_key"),
        upper(trim(col("iata_code"))).alias("origin_iata")
    )
)

destination_airport_lookup = (
    dim_airport
    .select(
        col("airport_key").alias("destination_airport_key"),
        upper(trim(col("iata_code"))).alias("destination_iata")
    )
)

route_lookup = (
    dim_route
    .select(
        "route_key",
        upper(trim(col("origin_airport"))).alias("route_origin"),
        upper(trim(col("destination_airport"))).alias("route_destination")
    )
)

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 5, Finished, Available, Finished, False)

In [4]:
fact_enriched = (
    flights.alias("f")

    .join(
        date_lookup.alias("d"),
        col("f.flight_date") == col("d.flight_date"),
        "left"
    )

    .join(
        airline_lookup.alias("a"),
        upper(trim(col("f.OP_UNIQUE_CARRIER")))
        == col("a.unique_carrier"),
        "left"
    )

    .join(
        origin_airport_lookup.alias("o"),
        upper(trim(col("f.ORIGIN")))
        == col("o.origin_iata"),
        "left"
    )

    .join(
        destination_airport_lookup.alias("da"),
        upper(trim(col("f.DEST")))
        == col("da.destination_iata"),
        "left"
    )

    .join(
        route_lookup.alias("r"),
        (
            upper(trim(col("f.ORIGIN")))
            == col("r.route_origin")
        )
        &
        (
            upper(trim(col("f.DEST")))
            == col("r.route_destination")
        ),
        "left"
    )

    .select(
        col("d.date_key"),
        col("a.airline_key"),
        col("o.origin_airport_key"),
        col("da.destination_airport_key"),
        col("r.route_key"),

        col("f.DEP_DELAY"),
        col("f.ARR_DELAY"),
        col("f.CANCELLED"),
        col("f.DIVERTED"),
        col("f.DISTANCE"),
        col("f.AIR_TIME")
    )
)

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 6, Finished, Available, Finished, False)

In [5]:
source_rows = flights.count()
enriched_rows = fact_enriched.count()

print("Source Rows:", source_rows)
print("Rows After Dimension Joins:", enriched_rows)
print("Row Difference:", enriched_rows - source_rows)

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 7, Finished, Available, Finished, False)

Source Rows: 547271
Rows After Dimension Joins: 547271
Row Difference: 0


In [6]:
key_validation = fact_enriched.agg(
    F.sum(
        F.when(col("date_key").isNull(), 1).otherwise(0)
    ).alias("null_date_keys"),

    F.sum(
        F.when(col("airline_key").isNull(), 1).otherwise(0)
    ).alias("null_airline_keys"),

    F.sum(
        F.when(col("origin_airport_key").isNull(), 1).otherwise(0)
    ).alias("null_origin_airport_keys"),

    F.sum(
        F.when(col("destination_airport_key").isNull(), 1).otherwise(0)
    ).alias("null_destination_airport_keys"),

    F.sum(
        F.when(col("route_key").isNull(), 1).otherwise(0)
    ).alias("null_route_keys")
)

display(key_validation)

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ac0839e-e5fe-4072-bbee-aac0a7118add)

In [7]:
print(
    "Distinct Airline Keys:",
    fact_enriched
    .select("airline_key")
    .distinct()
    .count()
)

fact_enriched.groupBy(
    "airline_key"
).count().orderBy(
    F.desc("count")
).show(20, truncate=False)

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 9, Finished, Available, Finished, False)

Distinct Airline Keys: 15
+-----------+------+
|airline_key|count |
+-----------+------+
|1          |115389|
|17         |77346 |
|16         |74384 |
|24         |58855 |
|54         |56814 |
|90         |22914 |
|76         |20750 |
|81         |20415 |
|80         |19580 |
|22         |17775 |
|68         |16972 |
|75         |16526 |
|86         |14379 |
|69         |8596  |
|14         |6576  |
+-----------+------+



In [8]:
flight_key_window = Window.orderBy(
    F.monotonically_increasing_id()
)

fact_flights_new = (
    fact_enriched
    .withColumn(
        "flight_key",
        F.row_number().over(flight_key_window)
    )
    .select(
        "flight_key",
        "date_key",
        "airline_key",
        "origin_airport_key",
        "destination_airport_key",
        "route_key",
        "DEP_DELAY",
        "ARR_DELAY",
        "CANCELLED",
        "DIVERTED",
        "DISTANCE",
        "AIR_TIME"
    )
)

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 10, Finished, Available, Finished, False)

In [9]:
print("Final Fact Rows:", fact_flights_new.count())

print(
    "Distinct Flight Keys:",
    fact_flights_new
    .select("flight_key")
    .distinct()
    .count()
)

print(
    "Null Airline Keys:",
    fact_flights_new
    .filter(col("airline_key").isNull())
    .count()
)

fact_flights_new.printSchema()

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 11, Finished, Available, Finished, False)

Final Fact Rows: 547271
Distinct Flight Keys: 547271
Null Airline Keys: 0
root
 |-- flight_key: integer (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- airline_key: integer (nullable = true)
 |-- origin_airport_key: integer (nullable = true)
 |-- destination_airport_key: integer (nullable = true)
 |-- route_key: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)



In [10]:
fact_flights_new.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_flights")

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 12, Finished, Available, Finished, False)

In [11]:
fact_flights_check = spark.table("fact_flights")

print("Persisted Fact Rows:", fact_flights_check.count())

print(
    "Persisted Distinct Flight Keys:",
    fact_flights_check
    .select("flight_key")
    .distinct()
    .count()
)

print(
    "Persisted Null Airline Keys:",
    fact_flights_check
    .filter(col("airline_key").isNull())
    .count()
)

print(
    "Persisted Distinct Airline Keys:",
    fact_flights_check
    .select("airline_key")
    .distinct()
    .count()
)

StatementMeta(, ac941942-c782-4062-a36b-1609069f1ac4, 13, Finished, Available, Finished, False)

Persisted Fact Rows: 547271
Persisted Distinct Flight Keys: 547271
Persisted Null Airline Keys: 0
Persisted Distinct Airline Keys: 15
